# Rubin Early Data Preview 2 (EDP2) — LSDB tutorial

In this tutorial, we will:
- Open the Rubin Data Preview 2 HATS catalog in LSDB
- Select specific columns for analysis
- Plot the catalog
- Select a region of the sky with a cone search
- Restrict the catalog by querying column values
- Map a function across the catalog
- Compute the catalog
- Write the catalog to disk
- Read the catalog from disk
- Cross-match the catalog with another catalog

If you just want a minimal code example, see the [**Starter Code**](https://docs.lsdb.io/en/latest/tutorials/pre_executed/rubin_dp2_starter.html) notebook.

Make sure you choose the **latest Weekly release** when you run this notebook on the RSP. This will give you the latest software versions with bug fixes and performance improvements.

## Setup

In [ ]:
import lsdb

import astropy.units as u
from astropy.coordinates import SkyCoord
from upath import UPath
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import logging

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("distributed").setLevel(logging.WARNING)

In [ ]:
# Setup
from dask.distributed import Client

client = Client(n_workers=4, memory_limit="4GiB", threads_per_worker=1)

We can keep an eye on the Dask dashboard to monitor our compute usage.

In [ ]:
print(f"Dask dashboard: {client.dashboard_link}")

### Further reading
[Tutorial: Setting up a Dask Client](https://docs.lsdb.io/en/latest/tutorials/dask_client.html)  
[Tutorial: Dask cluster configuration tips](https://docs.lsdb.io/en/latest/tutorials/dask-cluster-tips.html)

## Open catalog

First, let's open the Rubin DP2 catalog and plot its coverage.

In [ ]:
# Open DP2 catalog
# Path on RSP
base_path = UPath("/rubin/lsdb_data")
cat = lsdb.open_catalog(base_path / "object_collection")

In [ ]:
cat.plot_pixels();

### Further reading
[Tutorial: The Catalog Object](https://docs.lsdb.io/en/latest/tutorials/catalog_object.html)  
[Documentation: `Catalog`](https://docs.lsdb.io/en/latest/reference/catalog.html)

## Lazy evaluation

The catalog has been loaded *lazily*: no data has been read, only the catalog schema.

First we will filter the catalog down to only the data we need. Later, we will compute the catalog to download the data.

### Futher reading

Tutorial: **Why Lazy Evaluation?**  
[Tutorial: Lazy Operations in LSDB](https://docs.lsdb.io/en/latest/tutorials/lazy_operations.html)

## Choose columns

The full object catalog has more than a thousand columns. To keep things fast, LSDB opens only a curated **default subset** of columns unless you ask for more. For a description of the catalogs and their columns, see [About the EDP2 HATS Catalogs](https://docs.lsdb.io/en/latest/tutorials/rubin_dp2_release.html).

Let's take a look at the default columns that were loaded:

In [ ]:
cat.columns

We can request specific columns with `lsdb.open_catalog(..., columns=[...])`, or load every column with `columns="all"`. Note that `coord_ra` and `coord_dec` are always included.

In [ ]:
cat = lsdb.open_catalog(base_path / "object_collection", columns=["g_psfMag", "r_psfMag"])
cat

Choosing only these columns cut down the size of the catalog from 873.2 GB to 25.0 GB — less than 3%.

### Further reading
[Tutorial: column filtering](https://github.com/astronomy-commons/lsdb/blob/main/docs/tutorials/column_filtering.ipynb)  
[Documentation: `lsdb.open_catalog()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.open_catalog.html)


## Plot skymaps

We can see the spatial density of the catalog with a skymap:

In [ ]:
cat.plot_pixels();

We can also zoom in to a region in the sky. Let's focus near (ra=10, dec=-5).

In [ ]:
import astropy.units as u

fov = (8 * u.deg, 8 * u.deg)
center = SkyCoord(10 * u.deg, -5 * u.deg)
fig, ax = cat.plot_pixels(projection="AIT", fov=fov, center=center);

We can visualize the angular density in the same region:

In [ ]:
import hats

hats.inspection.plot_density(cat.hc_structure, ec="face", projection="AIT", fov=fov, center=center);

### Further reading

[Tutorial: Plotting Results](https://docs.lsdb.io/en/latest/tutorials/pre_executed/plotting.html)  
[Documentation: `lsdb.Catalog.plot_pixels()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.plot_pixels.html)  
[Documentation: `hats.inspection`](https://hats.readthedocs.io/en/stable/autoapi/hats/inspection/index.html#hats.inspection.plot_density)

## Spatial filtering

Now let's restrict the catalog to the region where we zoomed in.

In [ ]:
# Spatial filtering
cat = cat.cone_search(ra=10.0, dec=-5.0, radius_arcsec=2 * 3600)

In [ ]:
cat

The filtered catalog is significantly smaller: 

Catalog | Size
-------- | ------
Full DP2 | 873.2 GB
Selected columns | 25.0 GB
Selected columns and cone search | 5.5 MB

And it takes up only the selected region in the sky:

In [ ]:
cat.plot_pixels(projection="AIT", fov=fov, center=center);

### Further reading:

[Tutorial: Region Selection](https://docs.lsdb.io/en/latest/tutorials/region_selection.html)  
[Documentation: `lsdb.Catalog.cone_search()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.cone_search.html#lsdb.catalog.Catalog.cone_search)

## Query

We can further filter objects using a query based on the column values.

The query syntax is the same as for [`pandas.DataFrame.query()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html), except that local variables cannot be used.

In [ ]:
# Query to filter rows
cat = cat.query("g_psfMag < 28.0 and r_psfMag < 28.0")

Unlike spatial filtering, querying does not change the predicted size of the catalog. In reality, the catalog's size will decrease once we compute it.

In [ ]:
cat

### Further reading

[Tutorial: Row Filtering](https://docs.lsdb.io/en/latest/tutorials/row_filtering.html)  
[Documentation: `lsdb.Catalog.query()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.query.html#lsdb.catalog.Catalog.query)

## Mapping a function

We can map a function across all partitions (pixels) in the catalog using `lsdb.Catalog.map_partitions()`. From generating summary statistics per partition, to efficiently calculating transformations in parallel, this ability has many uses.

The mapping function takes in a dataframe that represents a partition—one row per object. It must return a dataframe. Some common usage patterns are:

- Add one or more columns to the dataframe and return it (e.g. calculate some attributes for each object, given the existing attributes). Rows still represent objects, and the number of rows remains the same.
- Return a subset of rows from the dataframe. Rows still represent objects, but the number of rows is less than the original partition.
- Return a dataframe of custom summary statistics for this partition (e.g. median value for each column). Rows no longer represent objects.

In [ ]:
# Map a function
def g_minus_r_mapper(df):
    df["g_minus_r_mag"] = df["g_psfMag"] - df["r_psfMag"]
    return df


cat = cat.map_partitions(g_minus_r_mapper)

### Further reading

[Tutorial: `map_partitions`](https://docs.lsdb.io/en/latest/tutorials/pre_executed/map_partitions.html)  
[Documentation: `lsdb.Catalog.map_partitions()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.map_partitions.html)

## Computing a catalog

All operations up to this point have been lazy, and have not downloaded data. To retrieve the data, we must **compute** the catalog.

First, we can use `lsdb.Catalog.head()` to retrieve only a few rows. This lets us make sure that everything looks right before executing a larger-scale computation of the whole catalog.

In [ ]:
cat.head(5)

We should now be able to see the task being executed in the Dask client. Check the Dask dashboard link near the beginning of this notebook and verify that you see activity.

Any time we compute the catalog, the result is a `nested_pandas.NestedFrame`, which is a kind of `pandas.DataFrame`.

In [ ]:
df = cat.head(5)
print(type(df))
import pandas as pd

print(isinstance(df, pd.DataFrame))

Everything looks right, so let's compute the whole catalog.

In [ ]:
# Compute catalog and write to disk
cat.write_catalog("dp2_example_cat", overwrite=True)

In [ ]:
# Compute catalog to dataframe
df = cat.compute()

In [ ]:
df

In [ ]:
# Read catalog from disk
cat = lsdb.open_catalog("dp2_example_cat")

### Further reading

[Documentation: `lsdb.Catalog.compute()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.compute.html)  
[Documentation: `lsdb.Catalog.write_catalog()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.write_catalog.html#lsdb.catalog.Catalog.write_catalog)  
[Documentation: `lsdb.open_catalog()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.open_catalog.html#lsdb.open_catalog)

## Crossmatching

We can combine two catalogs by [cross-matching](https://www.aanda.org/articles/aa/full_html/2017/11/aa30965-17/aa30965-17.html).

As an example, we'll cross-match the Rubin catalog with the [Dark Energy Survey (DES) catalog](https://data.lsdb.io/DES/DES_DR2_(US-East%2C_S3)).

In [ ]:
des_cat = lsdb.open_catalog("s3://stpubdata/mast/public/des/hats/des_dr2/", columns=["MAG_AUTO_G"])
des_cat

In [ ]:
des_cat.plot_pixels();

In [ ]:
# Crossmatch Rubin catalog with DES catalog
x_cat = cat.crossmatch(des_cat, suffix_method="overlapping_columns")

In [ ]:
x_cat.head()

Let's check that the cross-matched catalog covers the correct sky region. The cross-matched catalog should only contain the region where our filtered Rubin catalog overlaps the DES catalog.

In [ ]:
x_cat.plot_pixels(fov=fov, center=center);

There might also be extraneous matches, e.g. rows in the matched catalog that represent two different objects with similar coordinates, not one true object. As a quick check, the magnitudes reported from both catalogs should be close (differ by less than 1.0). Let's filter based on this criterion:

In [ ]:
# Crossmatch Rubin catalog with DES catalog
x_cat = cat.crossmatch(
    des_cat,
    suffix_method="overlapping_columns",
    # Filter out extraneous matches: match rows should have similar magnitude
).query("-1.0 < g_psfMag - MAG_AUTO_G < 1.0")

In [ ]:
x_cat.head()

In [ ]:
x_df = x_cat.compute()

In [ ]:
plt.figure(figsize=(14, 10))
plt.hist2d(x_df["g_psfMag"], x_df["MAG_AUTO_G"], bins=800, cmap="plasma")
plt.colorbar(label="density")
plt.xlabel("$g$ magnitude (Rubin)")
plt.ylabel("$g$ magnitude (DES)")
plt.title("$g$ magnitudes, Rubin vs DES");

### Further reading

[Tutorial: Astrometric epoch propagation](https://docs.lsdb.io/en/latest/tutorials/pre_executed/dp1-gaia-epoch-prop.html)  
[Documentation: `lsdb.Catalog.crossmatch()`](https://docs.lsdb.io/en/latest/reference/api/lsdb.catalog.Catalog.crossmatch.html#lsdb.catalog.Catalog.crossmatch)


## Close Dask client

In [ ]:
client.close()